# 01 — Data Loading & Validation

**Evaluation: Craft** — Clean, readable, well-structured code with extensible data loading.

This notebook validates the market data schema, handles granularity differences, and demonstrates the shared loader designed for extensibility (more zones, data sources, or live feeds).

In [6]:
import sys
sys.path.insert(0, '..')

import pandas as pd
from src.data_loader import (
    SPOT_ZONES,
    FLOW_ZONES,
    load_spot_price,
    load_total_load,
    load_generation,
    load_generation_pivot,
    load_flows_into,
    load_weather,
)

## Schema validation

In [8]:
def validate_zone(zone: str) -> dict:
    """Validate data availability and schema for a zone."""
    info = {"zone": zone, "ok": True, "issues": []}
    
    try:
        sp = load_spot_price(zone)
        info["spot_price"] = {"rows": len(sp), "range": (sp.index.min(), sp.index.max())}
    except Exception as e:
        info["ok"] = False
        info["issues"].append(f"spot_price: {e}")
    
    try:
        tl = load_total_load(zone)
        info["total_load"] = {"rows": len(tl), "range": (tl.index.min(), tl.index.max())}
    except Exception as e:
        info["issues"].append(f"total_load: {e}")
    
    try:
        gen = load_generation(zone)
        types = gen["type"].nunique()
        info["generation"] = {"rows": len(gen), "types": types}
    except Exception as e:
        info["issues"].append(f"generation: {e}")
    
    try:
        w = load_weather(zone)
        info["weather"] = {"rows": len(w), "cols": list(w.columns)}
    except Exception as e:
        info["issues"].append(f"weather: {e}")
    
    return info

for z in SPOT_ZONES:
    v = validate_zone(z)
    status = "✓" if v["ok"] else "✗"
    print(f"{status} {z}: spot={v.get('spot_price',{}).get('rows','?')} load={v.get('total_load',{}).get('rows','?')} gen_types={v.get('generation',{}).get('types','?')}")
    if v["issues"]:
        for i in v["issues"]:
            print(f"   └ {i}")

✓ AT: spot=24174 load=70176 gen_types=13
✓ BE: spot=24246 load=70176 gen_types=12
✓ CH: spot=17544 load=17543 gen_types=6
✓ CZ: spot=24174 load=57144 gen_types=15
✓ DE: spot=24174 load=70176 gen_types=17
✓ DK1: spot=24174 load=17534 gen_types=10
✓ FR: spot=24174 load=43874 gen_types=13
✓ NL: spot=24174 load=70176 gen_types=12
✓ PL: spot=24174 load=58368 gen_types=13


## Granularity & time alignment

Spot price: hourly until 2025-09-30, then 15-min (except CH). Total load & generation: 15-min. We resample for alignment.

In [9]:
# Spot price granularity change (2025-09-30 22:00 UTC)
sp_de = load_spot_price("DE")
cutoff = pd.Timestamp("2025-09-30 22:00", tz="UTC")
before = sp_de[sp_de.index < cutoff]
after = sp_de[sp_de.index >= cutoff]

print("DE Spot Price granularity:")
print(f"  Before 2025-09-30: {len(before)} points, freq ~{pd.Series(before.index).diff().median()}")
print(f"  After 2025-09-30:  {len(after)} points, freq ~{pd.Series(after.index).diff().median()}")

DE Spot Price granularity:
  Before 2025-09-30: 15334 points, freq ~0 days 01:00:00
  After 2025-09-30:  8840 points, freq ~0 days 00:15:00


In [10]:
# Resample to common frequency (15min) for analysis
def resample_spot_to_15min(df: pd.DataFrame) -> pd.DataFrame:
    """Forward-fill hourly spot to 15-min for alignment with load/generation."""
    return df.resample("15min").ffill()

sp_15 = resample_spot_to_15min(load_spot_price("DE"))
load = load_total_load("DE")
common_idx = sp_15.index.intersection(load.index)
print(f"Aligned DE data: {len(common_idx)} timestamps")
print(f"  Spot (resampled): {len(sp_15)}")
print(f"  Load: {len(load)}")

Aligned DE data: 70176 timestamps
  Spot (resampled): 70176
  Load: 70176


## Missing data handling

In [11]:
# Check for gaps and missing values
def check_gaps(df: pd.DataFrame, freq: str = "15min") -> pd.Series:
    full = pd.date_range(df.index.min(), df.index.max(), freq=freq, tz="UTC")
    missing = full.difference(df.index)
    return missing

load_de = load_total_load("DE")
gaps = check_gaps(load_de)
print(f"DE total-load: {len(load_de)} rows")
print(f"Expected (15min): {(load_de.index.max() - load_de.index.min()).total_seconds() / 900 + 1:.0f}")
print(f"Missing timestamps: {len(gaps)}")
if len(gaps) > 0 and len(gaps) <= 20:
    print(gaps.tolist())

DE total-load: 70176 rows
Expected (15min): 70176
Missing timestamps: 0


## Extensibility: loading multiple zones

In [12]:
from src.data_loader import load_all_spot_prices

all_prices = load_all_spot_prices()
print("Spot prices (wide):")
print(all_prices.head())
print(f"\nZones: {list(all_prices.columns)}")

Spot prices (wide):
                             AT    BE     CH    CZ    DE    DK1    FR    NL  \
time                                                                          
2024-01-01 00:00:00+00:00  0.01  0.01  21.99  0.01  0.01  28.14  0.01  0.01   
2024-01-01 01:00:00+00:00  0.02  0.00  14.32  0.02  0.00  26.66  0.00  0.00   
2024-01-01 02:00:00+00:00  0.00 -0.01  11.37  0.00 -0.01   4.14 -0.01 -0.01   
2024-01-01 03:00:00+00:00 -0.01 -0.03  11.35 -0.01 -0.03  -0.03 -0.03 -0.03   
2024-01-01 04:00:00+00:00 -0.01 -0.02  11.39 -0.01 -0.02  -0.02 -0.02 -0.02   

                              PL  
time                              
2024-01-01 00:00:00+00:00  74.50  
2024-01-01 01:00:00+00:00  73.29  
2024-01-01 02:00:00+00:00  71.58  
2024-01-01 03:00:00+00:00  73.27  
2024-01-01 04:00:00+00:00  74.32  

Zones: ['AT', 'BE', 'CH', 'CZ', 'DE', 'DK1', 'FR', 'NL', 'PL']
